# Lab 23: Multimodal AI: Text, Vision and Speech

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
This lab joins what you built in Labs 17 to 21 into one product: a **multi-modal assistant** that runs fully on your Jetson, like a small smart speaker with a camera and no cloud. *Multi-modal* means it works with several kinds of data (**modalities**): your **voice** (audio in), **text** (the language model), the **camera** (images) and its own **voice** (audio out). You say your **wake word** (Lab 19). The assistant **listens** to your question and writes it down (Lab 17), **thinks** about an answer with the language model (Lab 20) and **speaks** it (Lab 18). When you ask it to *look*, it takes a camera picture and answers with the vision model (Lab 21). You measure how long each step takes, find the slowest one and make the assistant faster.

**By the end of this lab you can:**
1. Name the modalities of the assistant, and join one small model for each into a pipeline.
2. Record a question until you stop speaking (end-of-speech detection).
3. Decide where to send a question (the language model or the camera model).
4. Measure the time of every stage and find the bottleneck.
5. Run the assistant, test it and make it better.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Cells that need your voice tell you when to speak. Press **Ctrl + S** often.

> **You need the work of the earlier labs:** the wake-word model of Lab 19 (openWakeWord), the Piper voice (Lab 18), the language model server (Lab 20) running on this Jetson **with its eyes** (Lab 21), if you want the camera questions. Part A checks them and tells you what is missing.
> **A speaker on the Jetson** is needed: the assistant runs in the background, and a background program cannot play sound in your browser. Connect the speaker to the Jetson (and set it with `labview.speaker_set(...)` if it is not the default, Lab 18).
> **Memory.** Four models are used (wake-word model, Whisper, the voice and the language model with its eyes, 3.5 GB). They fit in the 8 GB only when nothing else runs. Close all other notebooks (Kernel, Shut Down All Kernels) before you start.
> **Privacy.** The assistant listens all the time while it runs. Press **Stop** when you finish. Everything stays on the Jetson.

## 2. Background

**Modalities.** A *modality* is a kind of data: audio, text, image, video, sensor readings. A model that works with one kind is *uni-modal* (Whisper: audio to text; Piper: text to audio). A *vision-language model* (Lab 21) handles two kinds, image and text. A **multi-modal system** handles several kinds together. This assistant uses four: **audio in** (your question), **text** (the thinking), **image** (the camera) and **audio out** (the spoken answer).

There are two ways to build one. Some models take sound, pictures and text in **one** model, but then you cannot choose or swap the parts, and speech needs extra steps. So we build it as a **pipeline of small models**, one for each modality, and join them with a little code. This is how many real products are made.

**The pipeline.** The output of each step is the input of the next:

| Step | Job | Model | Lab |
|---|---|---|---|
| 1. Wake | Hear the wake word all the time, cheaply | Small wake-word model (openWakeWord) | 19 |
| 2. Listen | Record the question until you stop talking | Loudness detector | 17 and 19 |
| 3. Understand | Speech to text | Whisper | 17 |
| 4. Route | Does the question need the camera? | A few rules | this lab |
| 5. Think | Write the answer | Language model, or the same model with its eyes and a picture | 20, 21 |
| 6. Speak | Text to speech | Piper | 18 |

**Why a small model wakes a big one.** The wake-word model is tiny and runs all the time. Whisper and the language model are big, so they work only after the wake word. This saves memory, power and your privacy.

**End-of-speech detection.** The assistant does not know how long your question is. It records until you were **quiet for a moment** (for example 1 second) after you started to speak, or until a maximum time. A short quiet time reacts fast but can cut you off between words. A long one is safe but slow.

**Latency.** The time from the end of your question to the start of the answer is the **latency**. It is the sum of the stages:

`latency = speech to text + time to the first word of the model + the first sound of the voice`

People feel a delay below 1 second as natural and above 3 seconds as slow. The biggest stage is the **bottleneck**: improving anything else helps little.

**Routing.** A very simple router sends a question to the camera (the eyes of the model) when it contains a **look word** such as *see*, *look*, *camera*, *picture*, *holding*, *wearing*. Everything else goes to the language model. Real assistants use a model to decide, but rules are fast and easy to check.

**Staying inside the memory.** The assistant keeps a short history (as in Lab 20) so you can say "and the second one?". It is trimmed to stay inside the context window.

**Echo.** The microphone also hears the speaker. If the assistant listened while it spoke, it would answer itself. So it **stops listening while it speaks**.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(23)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Check and Load the Parts (40 min)

### Step 1: Settings
`WAKE_WORDS` is the pretrained wake word of Lab 19 (for example *hey jarvis*). Saying `STOP_WORD` in a question ends the conversation. `SILENCE` is the quiet time that ends your question. `MAX_QUESTION` is the longest question.

In [ ]:
# ---------------- Settings you can change ----------------
STT_SIZE = "base.en"          # the Whisper model of Lab 17 ("tiny.en" is faster)
LIMIT_DB = -38                # louder than this counts as speech (as in Labs 17 and 19)
SILENCE = 1.0                 # seconds of quiet that end your question
MAX_QUESTION = 8.0            # the longest question in seconds
WAKE_WORDS = ["hey_jarvis"]   # the pretrained wake word (Lab 19): hey_jarvis, alexa, hey_mycroft ...
STOP_WORD = "goodbye"         # say this word in a question to end the conversation
THRESHOLD = 0.50              # how sure the wake-word model must be (Lab 19)
COOLDOWN = 2.0                # seconds before the wake word is accepted again
SYSTEM = "You are a friendly voice assistant on a Jetson. Answer in one or two short sentences. Do not use lists or symbols."
LOOK_WORDS = ["see", "look", "camera", "picture", "holding", "wearing", "in front", "on the table", "colour", "color"]
# ----------------------------------------------------------
import csv
import glob
import hashlib
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime

import cv2
import numpy as np

import labview

RATE = 16000
print("This notebook is saved in:", os.getcwd())

### Step 2: Check Every Part
The cell checks each part and says what to do if it is missing. Do not go on until the first five lines say `OK`. The camera model is optional.

In [ ]:
def check_parts():
    status = []
    status.append(("Microphone", "OK" if shutil.which("arecord") else "missing: sudo apt install alsa-utils", "Lab 17"))
    try:
        import openwakeword
        status.append(("Wake-word model (openWakeWord)", "OK", "Lab 19"))
    except ImportError:
        status.append(("Wake-word model (openWakeWord)", "missing: do Part A of Lab 19", "Lab 19"))
    try:
        import faster_whisper
        status.append(("Whisper (faster-whisper)", "OK", "Lab 17"))
    except ImportError:
        status.append(("Whisper (faster-whisper)", "missing: pip install faster-whisper (see Lab 17)", "Lab 17"))
    voice = next((p for p in glob.glob("voices/*.onnx") + glob.glob("../lab18/voices/*.onnx")), None)
    status.append(("Piper voice", "OK " + voice if voice else "missing: do Part C of Lab 18", "Lab 18"))
    if labview.llm_ready():
        status.append(("Language model server", "OK", "Lab 20"))
        status.append(("Eyes (camera questions)", "OK" if labview.llm_vision() else "text only: the chosen student runs labview.llm_start(vision=True)", "Lab 21"))
    else:
        status.append(("Language model server", "not running: the chosen student runs labview.llm_start() (Part A of Lab 20)", "Lab 20"))
    for name, state, lab in status:
        print(f"{name:<36} {state}   ({lab})")
    return status, voice


status, VOICE_FILE = check_parts()

### Step 3: Load the Wake-Word Model (from Lab 19)
This is the same code as Lab 19: the `WakeDetector` that gives the pretrained openWakeWord model the sound in pieces of 80 ms (1280 numbers). The model files of your wake word are downloaded once.

In [ ]:
import openwakeword
from openwakeword.model import Model

for attempt in range(3):                               # a slow network can fail once; try again
    try:
        openwakeword.utils.download_models(WAKE_WORDS)
        break
    except Exception as error:
        print("Download problem, trying again:", str(error)[:80])
        time.sleep(2)

FRAME = 1280                                           # 80 ms of sound: the piece that the wake-word model reads


class WakeDetector:
    def __init__(self, model):
        self.model = model
        self.rest = np.zeros(0, np.int16)

    def feed(self, sound):
        """sound: numbers between -1 and 1. Returns {wake word: best score in this piece of sound}."""
        pcm = np.concatenate([self.rest, (np.clip(sound, -1, 1) * 32767).astype(np.int16)])
        best = {}
        whole = len(pcm) // FRAME * FRAME
        for start in range(0, whole, FRAME):
            for name, score in self.model.predict(pcm[start:start + FRAME]).items():
                best[name] = max(best.get(name, 0.0), float(score))
        self.rest = pcm[whole:]
        return best

    def reset(self):
        self.model.reset()
        self.rest = np.zeros(0, np.int16)


def level_db(x):
    return 20 * np.log10(max(float(np.sqrt(np.mean(np.square(x)))), 1e-6))


print("Wake words:", WAKE_WORDS, "| end word:", STOP_WORD)

### Step 4: Load Whisper, the Voice and the Language Model
The cell loads Whisper (it takes some seconds), checks the voice file and checks the language model server (the first question can be a little slower, as in Lab 20).

In [ ]:
from faster_whisper import WhisperModel

stt = WhisperModel(STT_SIZE, device="cpu", compute_type="int8", cpu_threads=4)
os.makedirs("speech_cache", exist_ok=True)
print("Whisper loaded.")

started = time.time()
labview.llm_chat([{"role": "user", "content": "Say ok."}], options={"max_tokens": 3})
print(f"Language model warm: {time.time() - started:.1f} s")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which parts say OK?:
- Which part is missing, if any, and what will you do?:
- Time to warm up the language model:


---
# Part B: Build the Steps One by One (50 min)

### Step 1: Speak (Lab 18)
`speak(text)` makes the sound with Piper (or takes it from the cache) and plays it on the Jetson speaker and in your browser. Test it.

In [ ]:
def speak(text, browser=True):
    """Text to speech with Piper. Returns the time until the sound started (the cache makes the repeated sentences instant)."""
    started = time.time()
    path = f"speech_cache/{hashlib.md5(text.encode()).hexdigest()[:12]}.wav"
    if not os.path.exists(path):
        command = [sys.executable, "-m", "piper", "-m", VOICE_FILE, "-f", path]
        result = subprocess.run(command, input=text.encode(), capture_output=True)
        if result.returncode != 0:
            raise SystemExit("Piper failed: " + result.stderr.decode(errors="replace")[-300:])
    made = time.time() - started
    labview.play(path, browser=browser)
    return made


print("Time to make the sound:", round(speak("Hello. I am your Jetson assistant."), 2), "seconds")

### Step 2: Listen until You Stop (Lab 17)
`listen_until_silence` reads the microphone stream every 0.1 seconds. It waits for speech to begin, and ends the question when `SILENCE` seconds of quiet follow or `MAX_QUESTION` seconds have passed. It gives back the recorded sound. The cell checks it with a fake stream first.

In [ ]:
def listen_until_silence(stream, wait_for_speech=5.0, silence=None, max_seconds=None, now=time.time, sleep=time.sleep):
    """Record from the microphone stream until you were quiet for `silence` seconds after speaking. Returns the sound ('' if nobody spoke)."""
    silence = SILENCE if silence is None else silence
    max_seconds = MAX_QUESTION if max_seconds is None else max_seconds
    chunks, spoke, last_loud, started = [], False, 0.0, now()
    while now() - started < max_seconds + wait_for_speech:
        sleep(0.1)
        piece = stream.latest(0.1)
        chunks.append(piece)
        if level_db(piece) > LIMIT_DB:
            if not spoke:
                spoke, begin = True, now()
            last_loud = now()
        if spoke and now() - last_loud >= silence:
            break
        if spoke and now() - begin >= max_seconds:
            break
        if not spoke and now() - started >= wait_for_speech:
            return np.zeros(0, np.float32)
    return np.concatenate(chunks) if spoke else np.zeros(0, np.float32)


class FakeStream:
    """A fake microphone for a test: quiet, then 1.5 seconds of speech, then quiet again."""
    def __init__(self):
        self.t0 = time.time()

    def latest(self, seconds):
        t = time.time() - self.t0
        loud = 0.5 < t < 2.0
        return (np.random.randn(int(seconds * RATE)) * (0.1 if loud else 0.0005)).astype(np.float32)


started = time.time()
sound = listen_until_silence(FakeStream())
print(f"Recorded {len(sound) / RATE:.1f} s of sound in {time.time() - started:.1f} s (expected: speech ends at 2.0 s, plus {SILENCE} s of quiet)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the fake test stop about `SILENCE` seconds after the speech?:
- What happens if `SILENCE` is 0.3? What happens if it is 3?:


### Step 3: Understand (Lab 17)
`transcribe` is the speech to text of Lab 17. It first checks for speech so that Whisper does not invent words from silence.

In [ ]:
def transcribe(x):
    """Speech to text. Returns (text, seconds). An empty sound gives an empty text."""
    started = time.time()
    if len(x) < RATE // 4:
        return "", 0.0
    segments, _ = stt.transcribe(x.astype(np.float32), language="en", beam_size=1)
    return " ".join(s.text.strip() for s in segments).strip(), time.time() - started

### Step 4: Route and Think (Labs 20 and 21)
`needs_camera(text)` is the router. `grab_frame()` takes one picture from the camera (it opens the camera, throws away 5 pictures so the exposure settles, and lets the camera go again). `think(text)` returns the answer, the route and the time to the first word. The router is checked first.

In [ ]:
def needs_camera(text):
    """True when the question contains a look word."""
    spoken = " ".join(re.findall(r"[a-z]+", text.lower()))
    return any(word in spoken for word in LOOK_WORDS)


for test in ("What do you see?", "What is the capital of Malaysia?", "What am I holding?", "Tell me a joke", "What colour is my shirt?"):
    print(f"{test!r:36} -> {'CAMERA' if needs_camera(test) else 'language model'}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Is every question routed as you expect?:
- Find a question that the router sends the wrong way. What would you change?:


In [ ]:
history = [{"role": "system", "content": SYSTEM}]


def trim(messages, budget=1200):
    messages = list(messages)
    while sum(len(m["content"]) for m in messages) // 4 > budget and len(messages) > 2:
        del messages[1]
    return messages


def grab_frame():
    """One picture from the camera, or None if the camera is busy or missing."""
    try:
        camera = labview.open_camera()
    except SystemExit:
        return None
    for _ in range(5):
        camera.read()
    ok, frame = camera.read()
    camera.release()
    return frame if ok else None


def think(text):
    """Answer a question. Returns (answer, route, seconds to the first word)."""
    global history
    if needs_camera(text):
        frame = grab_frame()
        if frame is None:
            return "I cannot use the camera now. Please stop other programs that use it.", "camera-error", 0.0
        jpeg = cv2.imencode(".jpg", cv2.resize(frame, (640, 480)))[1].tobytes()
        result = labview.llm_chat([{"role": "user", "content": text}], images=[jpeg], options={"temperature": 0.2, "max_tokens": 80})
        return result["text"], "camera", result["first_token_s"]
    history.append({"role": "user", "content": text})
    history = trim(history)
    result = labview.llm_chat(history, options={"temperature": 0.5, "max_tokens": 100})
    history.append({"role": "assistant", "content": result["text"]})
    return result["text"], "language", result["first_token_s"]


answer, route, first = think("In one sentence, what is a Jetson?")
print(f"[{route}, first word after {first:.1f} s] {answer}")

### Step 5: Check the Whole Chain by Typing
Before you use your voice, test steps 4 to 6 with a typed question. The cell asks, then speaks the answer and prints the time of each stage.

In [ ]:
question = input("Type a question: ").strip() or "What is the tallest animal?"
t0 = time.time()
answer, route, first = think(question)
t1 = time.time()
made = speak(answer)
print(f"Route: {route}\nAnswer: {answer}")
print(f"Think: {t1 - t0:.1f} s (first word {first:.1f} s) | Voice made in {made:.1f} s | Total {time.time() - t0:.1f} s")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Route and answer:
- Time of thinking and of making the voice:
- Which of the two is slower?:


---
# Part C: Run the Assistant (40 min)

### Step 1: The Assistant
The class `Assistant` runs in a background thread and ties all the steps together:

1. **Wait** for the wake word (the `WakeDetector` of Lab 19: it reads the sound in pieces of 80 ms and compares the score with `THRESHOLD`).
2. A short **beep** says "I am listening".
3. **Listen** until you stop speaking, then **transcribe**.
4. If you say the end word, the assistant says goodbye and stops.
5. **Think** (language or camera model) and **speak**.
6. Write the times of every stage in `assistant_log.csv`.

While it speaks, it does not listen (no echo). The window under the cell shows the conversation. Press **Stop** to end it.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def beep():
    t = np.arange(int(0.15 * RATE)) / RATE
    labview.play((0.3 * np.sin(2 * np.pi * 880 * t) * np.hanning(len(t))).astype(np.float32), browser=False)


def log_turn(heard, answer, route, times):
    first = not os.path.exists("assistant_log.csv")
    with open("assistant_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "heard", "route", "answer", "listen_s", "stt_s", "think_first_word_s", "think_total_s", "voice_s", "latency_s"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), heard, route, answer] + [round(times[k], 2) for k in
                         ("listen", "stt", "first", "think", "voice", "latency")])


class Assistant:
    def __init__(self):
        self.stream = labview.MicStream(RATE)
        self.detector = WakeDetector(Model(wakeword_models=WAKE_WORDS, inference_framework="onnx"))
        self.last_time = time.time()
        self.last_wake = 0.0
        self.stop_flag = threading.Event()
        self.lines = ["Say the wake word ..."]
        self.screen = widgets.HTML("Say the wake word ...")
        self.button = widgets.Button(description="Stop", button_style="danger")
        self.button.on_click(lambda _: self.stop())
        display(widgets.VBox([self.button, self.screen]))
        threading.Thread(target=self._run, daemon=True).start()

    def show(self, line):
        self.lines = (self.lines + [line])[-8:]
        self.screen.value = "<br>".join(self.lines)

    def heard_word(self):
        """One step of the wake-word listener (Lab 19). Returns the wake word, or None."""
        now = time.time()
        seconds = min(now - self.last_time, 2.0)
        self.last_time = now
        scores = self.detector.feed(self.stream.latest(seconds))
        for word, score in scores.items():
            if score >= THRESHOLD and now - self.last_wake >= COOLDOWN:
                self.last_wake = now
                return word
        return None

    def _run(self):
        try:
            while not self.stop_flag.is_set():
                time.sleep(0.05)
                if not self.stream.alive():
                    self.show("The microphone stopped: " + self.stream.error())
                    return
                if self.heard_word():
                    self.turn()
                    self.detector.reset()                                  # ignore the echo of our own voice
                    self.last_time, self.last_wake = time.time(), time.time()
        except SystemExit as error:
            self.show("Error: " + str(error)[:150])

    def turn(self):
        beep()
        self.show("<b>Listening ...</b>")
        t0 = time.time()
        sound = listen_until_silence(self.stream)
        t1 = time.time()
        heard, stt_seconds = transcribe(sound)
        t2 = time.time()
        if not heard:
            self.show("(I did not hear a question)")
            return
        self.show("You: " + heard)
        if STOP_WORD in re.findall(r"[a-z]+", heard.lower()):
            speak("Goodbye.", browser=False)
            self.stop()
            return
        answer, route, first = think(heard)
        t3 = time.time()
        self.show(f"Assistant ({route}): {answer}")
        voice_seconds = speak(answer, browser=False)
        times = {"listen": t1 - t0, "stt": stt_seconds, "first": first, "think": t3 - t2, "voice": voice_seconds,
                 "latency": (t3 - t1) + voice_seconds}
        log_turn(heard, answer, route, times)
        self.show(f"<i>latency {times['latency']:.1f} s = speech to text {stt_seconds:.1f} + thinking {t3 - t2:.1f} + voice {voice_seconds:.1f}</i>")

    def stop(self):
        self.stop_flag.set()
        self.stream.stop()
        self.show("[stopped]")


print("The assistant class is ready.")

### Step 2: Talk to It
Run the cell. Say the **wake word**, wait for the beep and ask a question. Try: *What is the capital of Malaysia?*, *Tell me a short joke*, *What do you see?* (camera), and at the end say the **end word**. Press **Stop** if it does not stop. The window shows the conversation and the **latency** of each answer.

In [ ]:
assistant = Assistant()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the wake word work? How many tries out of 5?:
- Questions you asked and whether the answers were good:
- The latency of three answers (from the window):
- Which stage was the slowest (speech to text, thinking or voice)?:


After you stopped, see the log:

In [ ]:
if os.path.exists("assistant_log.csv"):
    with open("assistant_log.csv") as f:
        print(f.read())
else:
    print("No turns yet.")

---
# Part D: Find the Bottleneck and Make It Better (30 min)

### Step 1: Where Does the Time Go?
The cell reads `assistant_log.csv` and shows the **average time of each stage**. The biggest one is the bottleneck.

In [ ]:
def stage_report():
    with open("assistant_log.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        print("No turns in the log.")
        return
    stages = [("speech to text", "stt_s"), ("thinking (total)", "think_total_s"), ("voice", "voice_s"), ("latency", "latency_s")]
    print(f"Turns: {len(rows)}")
    for name, key in stages:
        values = [float(r[key]) for r in rows]
        print(f"  {name:<18} average {sum(values) / len(values):5.2f} s   largest {max(values):5.2f} s   " + "#" * int(3 * sum(values) / len(values)))


if os.path.exists("assistant_log.csv"):
    stage_report()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average time of each stage:
- The bottleneck and why you think it is slow:
- What could you change to make it faster? (Think of the model size, the answer length and the silence time.):


### Step 2: Make It Better
Choose **at least one improvement**, change the code or the settings, and run the assistant again. Compare the latency before and after with `stage_report()`.

| Change | Effect |
|---|---|
| `STT_SIZE = "tiny.en"` | Faster speech to text, a bit less accurate |
| `"max_tokens": 60` in `think` and a shorter system prompt | Shorter answers, faster |
| `SILENCE = 0.7` | The question ends sooner |
| A smaller model (ask your teacher to start the server with `lfm2.5-vl-1.6b`, which is smaller and faster) | Faster thinking, simpler answers |
| Speak the **first sentence** while the rest is still written | The answer starts sooner (needs a change in `think`) |
| More look words, or a **second wake action** (for example the end word to mute) | Better routing and control |
| Add **tools**: the time (`datetime.now()`), a calculator, or an MQTT message to your phone (Lab 16) | The assistant can do things, not only talk |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Latency before and after:
- Did the quality of the answers change?:


---
# Part E: Test and Present (20 min)

### Step 1: Test
Test each situation and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | Wake word, then a normal question, 30 cm from the microphone | Answer within about 5 seconds | |
| 2 | Wake word in a noisy room | Works or false wake-ups? | |
| 3 | A long question (about 10 seconds) | Cut at `MAX_QUESTION` | |
| 4 | Wake word and then silence | "I did not hear a question" | |
| 5 | A follow-up: "What is the capital of Japan?" then "And of Korea?" | Uses the history | |
| 6 | "What do you see?" | The camera route | |
| 7 | Say the end word | Goodbye and stop | |
| 8 | The assistant speaks: does it wake itself? | No | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4:
- Test 5 to 8:
- Where did the assistant fail? What would you do about it?:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: wake the assistant, ask a question and a camera question, show the latency of each stage and the bottleneck, and show the improvement.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The wake word and the models used:
- Average latency and the bottleneck:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement and its effect:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Congratulations.** You built a private multi-modal assistant (voice in, text, camera and voice out) from a pretrained wake-word model, Whisper, a language model with eyes and a neural voice, all on a 15-watt board. The remaining labs apply AI in real-world products: a smart box that reacts and reports, and your own capstone project.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The part check says OK for every needed part, and you name the modality of each model | |
| 2 | The assistant speaks a text and `listen_until_silence` passes the fake test | |
| 3 | The router sends the test questions to the right model | |
| 4 | The typed question is answered and spoken, with the time of each stage | |
| 5 | The assistant wakes with the wake word, listens, answers and speaks | |
| 6 | A camera question ("What do you see?") is answered | |
| 7 | The stage report is read and the bottleneck is named | |
| 8 | At least one improvement is made and the latency is compared | |
| 9 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Name the six steps of the assistant, the model of each one, and the modality it takes in and gives out (audio, text or image).:
- 2. Why does a small wake-word model wake the big models and not the other way round?:
- 3. How does the program know that your question has ended? What is the cost of a very short quiet time?:
- 4. What is the latency and what does it consist of? What is a bottleneck?:
- 5. Why does the assistant stop listening while it speaks?:
- 6. How does the router decide to use the camera? Give a question where it is wrong.:
- 7. Name two ways to make the assistant answer sooner.:
- 8. Why is it a benefit that everything runs on the Jetson? Name one cost.:
- 9. Which is the best next feature? (a) Let the assistant switch a lamp on by voice (b) A bigger screen (c) A louder beep (d) A longer wake word:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| A part says `missing` in Part A | An earlier lab was not done on this account | Do the lab that is named at the end of the line |
| `No module named 'openwakeword'` | Lab 19 was not done on this account | Do Part A of Lab 19 (it installs openWakeWord) |
| The wake word never works | The score stays below `THRESHOLD`, the microphone is wrong, or you are too far | Test it again with the cells of Lab 19. Lower `THRESHOLD` to 0.3. Speak closer. |
| It wakes up by itself | `THRESHOLD` too low, or its own voice is heard | Raise `THRESHOLD` (0.7). Lower the speaker volume. |
| `I did not hear a question` | You spoke too late or `LIMIT_DB` is too high | Speak right after the beep. Lower `LIMIT_DB` a little. |
| The question is cut in the middle | `SILENCE` is too short | Raise `SILENCE` to 1.5 |
| It waits a long time after the question | `SILENCE` is long, or the answer is long | Lower `SILENCE`. Use a shorter system prompt and `max_tokens`. |
| `Cannot reach the language model` or an out-of-memory error | The server stopped, or too many programs | Run `labview.llm_start()` again (one student). Close other notebooks. Kernel, Shut Down All Kernels. |
| `Piper failed` | The voice file is missing | Do Part C of Lab 18 |
| `I cannot use the camera now` | Another program uses the camera | Press Stop in other live cells, shut down other kernels |
| The microphone stopped | Another notebook records | Stop the other recording cells |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(23)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Home Assistant, Assist (an open voice assistant): https://www.home-assistant.io/voice_control/
- Whisper: https://github.com/openai/whisper
- Piper: https://github.com/OHF-Voice/piper1-gpl
- llama.cpp: https://github.com/ggml-org/llama.cpp